<a href="https://colab.research.google.com/github/pratripat/English-to-Hindi-Translator/blob/main/EngToHindi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Pipeline

## Step 1: Load raw corpus

In [1]:
from datasets import load_dataset

In [2]:

# IITB English-Hindi parallel corpus, standard benchmark for this task.
# ~1.66M train pairs, ~2500 val, ~2500 test.
raw = load_dataset("cfilt/iitb-english-hindi")

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [3]:
print(raw['train'][0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


## Step 2: Clean and filter the corpus to something that can be used

In [4]:
import re
from tqdm import tqdm

In [5]:
# Cleans a given text
def basic_clean(text: str) -> str:
  text = text.strip() # Remove trailing white spaces
  text = re.sub(r"\s+", " ", text) # This collapses multiple spaces, tabs into one space. This helps the model retain context or make multiple tokens for the spaces and tabs.
  return text

In [6]:
# Filters out the pairs that are actually relevant
def filter_pair(en: str, hi: str, min_len=1, max_len=40, max_ratio=2.5):
  en_toks = en.split()
  hi_toks = hi.split()

  # There should be atleast one token that will be converted into hindi tokens.
  # Max length is kept because the conversion is a costly task O(n^2)
  if not en_toks or not hi_toks:
      return False
  if len(en_toks) < min_len or len(hi_toks) < min_len:
      return False
  if len(en_toks) > max_len or len(hi_toks) > max_len:
      return False

  # When some english sentence is converted into another language, it is usually relatively similar size
  # in terms of tokens, if we have 50 words being converted to 3 words, then the conversion is probably wrong
  # the data fetching might have problems or the data set might have some irrelevant conversions
  # (sentence alignment error)
  ratio = len(en_toks) / len(hi_toks)
  if ratio > max_ratio or ratio < (1 / max_ratio):
      return False

  return True

In [7]:
def clean_and_filter_split(split, max_len=40, max_ratio=2.5):
  en_lines = []
  hi_lines = []
  seen = set()

  for ex in tqdm(split, desc='cleaning'):
    en = basic_clean(ex['translation']['en'])
    hi = basic_clean(ex['translation']['hi'])

    if not filter_pair(en, hi, max_len=max_len, max_ratio=max_ratio):
      continue

    if (en, hi) in seen:
      continue

    seen.add((en, hi))
    en_lines.append(en)
    hi_lines.append(hi)

  return en_lines, hi_lines

In [8]:
DEV_SUBSET_SIZE = 100000 # Starting small just for pipelining sake

In [9]:
train_subset = raw["train"].select(range(min(DEV_SUBSET_SIZE, len(raw["train"]))))

train_en, train_hi = clean_and_filter_split(train_subset)
val_en, val_hi = clean_and_filter_split(raw['validation'])
test_en, test_hi = clean_and_filter_split(raw['test'])

cleaning: 100%|██████████| 2507/2507 [00:00<00:00, 18055.37it/s]


In [10]:
print(f"train pairs kept: {len(train_en)}")
print(f"val pairs kept:   {len(val_en)}")
print(f"test pairs kept:  {len(test_en)}")

train pairs kept: 21588
val pairs kept:   508
test pairs kept:  2296


## Step 3: Writing the data into plain files for SentencePiece training

In [11]:
import os

In [12]:
os.makedirs("data", exist_ok=True)

def write_lines(lines, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

write_lines(train_en, "data/train.en")
write_lines(train_hi, "data/train.hi")
write_lines(val_en, "data/val.en")
write_lines(val_hi, "data/val.hi")
write_lines(test_en, "data/test.en")
write_lines(test_hi, "data/test.hi")

## Step 4: Training the tokenizers

In [13]:
import sentencepiece as spm

We will be using two different vocabs for english and hindi. This is because we will never encounter a scenario where we need both the english and hindi contents together. So there is no point forcing both the tokens to be in one table. The encoder uses the english vocab and the decoder uses the hindi vocab (for generation) along with the english vocab.

In [14]:
VOCAB_SIZE_EN = 8000
VOCAB_SIZE_HI = 8000

In [15]:
def train_sentencepiece(input_file, model_prefix, vocab_size):
    spm.SentencePieceTrainer.train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe", # byte pair encoding
        character_coverage=0.9995,  # important for Hindi: keeps rare Devanagari chars
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
    )

In [16]:
train_sentencepiece("data/train.en", "data/spm_en", VOCAB_SIZE_EN)
train_sentencepiece("data/train.hi", "data/spm_hi", VOCAB_SIZE_HI)

In [17]:
sp_en = spm.SentencePieceProcessor(model_file="data/spm_en.model")
sp_hi = spm.SentencePieceProcessor(model_file="data/spm_hi.model")

In [18]:
# Sanity check -- always eyeball this before trusting the tokenizer
sample_en = train_en[0]
sample_hi = train_hi[0]
print("EN:", sample_en)
print("EN tokens:", sp_en.encode(sample_en, out_type=str))
print("HI:", sample_hi)
print("HI tokens:", sp_hi.encode(sample_hi, out_type=str))

EN: Give your application an accessibility workout
EN tokens: ['▁Give', '▁your', '▁application', '▁an', '▁accessibility', '▁work', 'out']
HI: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
HI tokens: ['▁अपने', '▁अनुप्रयोग', '▁को', '▁पहुंचनीयता', '▁व्या', 'या', 'म', '▁का', '▁ला', 'भ', '▁दें']


The tokens are being created properly (as it should have)

## Step 5: Dataset

In [19]:
import torch
from torch.utils.data import Dataset

In [20]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
MAX_SEQ_LEN = 64

In [21]:
# Pre-tokenize everything, avoids retokenizing every epoch and lazy tokenizes in __getitem__
class TranslationDataset(Dataset):
  def __init__(self, en_lines, hi_lines, sp_en, sp_hi, max_len=MAX_SEQ_LEN):
    assert len(en_lines) == len(hi_lines)
    self.max_len = max_len

    self.en_ids = []
    self.hi_ids = []

    for en, hi in zip(en_lines, hi_lines):
      en_ids = sp_en.encode(en, out_type=int)[: max_len - 1] + [EOS_ID]
      # Decoder input needs BOS_ID token at the start, target needs EOS_ID token at the end which is added later
      hi_ids = sp_hi.encode(hi, out_type=int)[: max_len - 2]

      self.en_ids.append(en_ids)
      self.hi_ids.append(hi_ids)

  def __len__(self):
    return len(self.en_ids)

  def __getitem__(self, idx):
    return self.en_ids[idx], self.hi_ids[idx]

In [22]:
train_dataset = TranslationDataset(train_en, train_hi, sp_en, sp_hi)
val_dataset = TranslationDataset(val_en, val_hi, sp_en, sp_hi)

In [23]:
print(f'train size: {len(train_dataset)}')
print(f'val size: {len(val_dataset)}')

train size: 21588
val size: 508


## Step 6: Adding paddings and masks

In [24]:
from torch.nn.utils.rnn import pad_sequence

In [25]:
def collate_fn(batch):
  # batch: list of (en_ids, hi_ids) tuples of variable lengths

  en_batch, hi_batch = zip(*batch)
  en_tensors = [torch.tensor(x, dtype=torch.long) for x in en_batch]
  dec_in_tensors = [torch.tensor([BOS_ID], dtype=torch.long) + torch.tensor(x, dtype=torch.long) for x in hi_batch]
  dec_tgt_tensors = [torch.tensor(x, dtype=torch.long) + torch.tensor([EOS_ID], dtype=torch.long) for x in hi_batch]

  en_padded = pad_sequence(en_tensors, batch_first=True, padding_value=PAD_ID) # (B, src_len)
  dec_in_padded = pad_sequence(dec_in_tensors, batch_first=True, padding_value=PAD_ID) # (B, tgt_len)
  dec_out_padded = pad_sequence(dec_tgt_tensors, batch_first=True, padding_value=PAD_ID) # (B, tgt_len)

  # Used in attention to prevent tokens from attending to pad positions
  src_padding_mask = (en_padded != PAD_ID)
  tgt_padding_mask = (dec_in_padded != PAD_ID)

  # NOTE: the causal (look-ahead) mask for decoder self-attention is NOT
  # built here -- it doesn't depend on the data, only on tgt_len, so it
  # belongs in the model/training loop where you can build it once per
  # batch shape (or even cache it), not in the dataloader.

  return {
      "src": en_padded,
      "tgt_in": dec_in_padded,
      "tgt_out": dec_out_padded,
      "src_padding_mask": src_padding_mask,
      "tgt_padding_mask": tgt_padding_mask,
  }

## Step 7: Length bucketed batch sampler

Naive random batching would mix short and long line sequences together into one batch. This would cause many pad tokens to be used in the shorter tokens wasting compute time for the model. So here bucketed sorts by length first, so batches are made up of similar sized sentences to avoid alot of padding.

In [26]:
import random
from torch.utils.data import Sampler

In [27]:
class BucketBatchSampler(Sampler):
  def __init__(self, dataset, batch_size, bucket_size_multiplier=100):
    self.dataset = dataset
    self.batch_size = batch_size
    self.bucket_size = batch_size * bucket_size_multiplier

  def __iter__(self):
    indices = list(range(len(self.dataset)))
    random.shuffle(indices)

    # Sort within buckets with src + tgt length to add some randomness to it so that very similar length sentences also do not come at once
    batches = []
    for i in range(0, len(indices), self.bucket_size):
      bucket = indices[i : i + self.bucket_size]
      bucket.sort(key=lambda idx: len(self.dataset[idx][0]) + len(self.dataset[idx][1]))

      for j in range(0, len(bucket), self.batch_size):
        batches.append(bucket[j : j + self.batch_size])

    random.shuffle(batches) # Shuffling so that the model does not receive lengths of sentences slowly increasing

    return iter(batches)

  def __len__(self):
    return (len(self.dataset) + self.batch_size - 1) // self.batch_size

## Step 8: Data loaders

In [28]:
from torch.utils.data import DataLoader

In [29]:
BATCH_SIZE = 64

In [30]:
train_sampler = BucketBatchSampler(train_dataset, BATCH_SIZE)
train_loader = DataLoader(
  train_dataset,
  batch_sampler=train_sampler,
  collate_fn=collate_fn,
  num_workers=2
)

val_sampler = BucketBatchSampler(val_dataset, BATCH_SIZE)
val_loader = DataLoader(
  val_dataset,
  batch_sampler=val_sampler,
  collate_fn=collate_fn,
  num_workers=2
)

In [31]:
# Quick sanity check
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

src torch.Size([64, 4]) torch.int64
tgt_in torch.Size([64, 3]) torch.int64
tgt_out torch.Size([64, 3]) torch.int64
src_padding_mask torch.Size([64, 4]) torch.bool
tgt_padding_mask torch.Size([64, 3]) torch.bool


# Model architecture

In [32]:
import math
import torch
import torch.nn as nn

## Positional Encoding
Using sinusoidal positional encoding instead of learned encodings.Just following the 'Attention is all you need' paper convention, although using learned encodings will not make much difference.

Here this just helps speed up the training time.

Sinusoidal encodings work because for any fixed offset k, PE(pos + k) can be expressed as a linear function of PE(pos). This helps the model to attend by "relative" position, not just absolute position

In [33]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=512, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)

    # div_term: 1 / 10000^(2i/d_model), computed in log-space for numerical stability
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0) # (1, max_len, d_model) -> batch dim for broadcasting
    self.register_buffer('pe', pe) # not a parameter

  def forward(self, x):
    # x shape - (B, seq_len, d_model)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

## Multi Head Attention
(This will be used for self attention and cross attention)

Same module handles encoder self attention, decoder masked self attention and decoder cross attention. The only different is the Q, K, V that are passed into the function

In [34]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, dropout=0.1):
    super().__init__()
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.w_q = nn.Linear(d_model, d_model)
    self.w_k = nn.Linear(d_model, d_model)
    self.w_v = nn.Linear(d_model, d_model)
    self.w_o = nn.Linear(d_model, d_model) # output projection after concatenating heads

    self.dropout = nn.Dropout(dropout)

  def split_heads(self, x, batch_size):
    # (B, seq_len, d_model) -> (B, num_heads, seq_len, d_k)
    x = x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
    return x

  def forward(self, query, key, value, mask=None):
    # query, key, value -> (B, seq_len, d_model)
    #   self attention: query = key = value = same tensor
    #   cross attention: query = decoder states, key = value = encoder output

    batch_size = query.size(0)

    Q = self.split_heads(self.w_q(query), batch_size) # (B, h, seq_len, d_k)
    K = self.split_heads(self.w_k(key), batch_size) # (B, h, seq_len, d_k)
    V = self.split_heads(self.w_v(value), batch_size) # (B, h, seq_len, d_k)

    # Scaled dot product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k) # (B, h, seq_len_q, seq_len_k)

    if mask is not None:
      scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = torch.softmax(scores, dim=-1) # (B, h, seq_len_q, seq_len_k)
    attn_weights = self.dropout(attn_weights)

    context = torch.matmul(attn_weights, V) # (B, h, seq_len, d_k)

    # concatenate heads back together
    context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model) # (B, seq_len, d_model)

    return self.w_o(context)

## Positionwise Feed Forward

In [35]:
class PositionwiseFeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()

    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)
    self.relu = nn.ReLU()

  def forward(self, x):
    return self.linear2(self.dropout(self.relu(self.linear1(x))))

## Encoder layer / block

_(self-attention -> add&norm -> feedfoward -> add&norm)_

Ideally pre-norm is preferred as it helps ease out the residual connections.

* In post-norm, the residual stream keeps getting normalized at every layer/block, which means gradients passing back have to go through many LayerNorm operations. This makes deep post-norm models notoriously hard to train without careful learning rate warmup.
* In pre-norm, the residual connections are clean, unimpeded sum of all sublayer outputs, gradients can flow straight back through it without repeatedly passing through normalization, which makes training much more stable at depth and generally forgiving of learning rate choices.

So, I will be using the pre-norm for the following reasons

_(add&norm -> self-attention -> add&norm -> feedfoward)_

In [36]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, src_mask):
    residual = x
    x_norm = self.norm1(x)
    attn_out = self.self_attn(x_norm, x_norm, x_norm, src_mask)

    x = residual + self.dropout(attn_out)

    residual = x
    x_norm = self.norm2(x)
    ff_out = self.feed_forward(x_norm)

    x = residual + self.dropout(ff_out)

    return x

## Decoder Layer/Block

_(masked self attn -> add&norm -> cross attn -> add&norm -> feedforward -> add&norm)_

Here too, after the pre-norm step, the architecture changes to:

_(add&norm -> masked self attn -> add&norm -> cross attn -> add&norm -> feedfoward)_

In [37]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, enc_output, src_mask, tgt_mask):
    # Self Attention uses the q = k = v = decoder states and the causal mask filters the future tokens out
    residual = x
    x_norm = self.norm1(x)
    self_attn_out = self.self_attn(x_norm, x_norm, x_norm, tgt_mask)
    x = residual + self.dropout(self_attn_out)

    # Cross Attention uses q = decoder states, k = v = encoder output
    residual = x
    x_norm = self.norm2(x)
    cross_attn_out = self.cross_attn(x_norm, enc_output, enc_output, src_mask)
    x = residual + self.dropout(cross_attn_out)

    # Feed forward
    residual = x
    x_norm = self.norm3(x)
    ff_out = self.feed_forward(x_norm)
    x = residual + self.dropout(ff_out)

    return x

## Encoder (Stack of N Encoder Layers/Blocks)

In [38]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
    super().__init__()

    self.d_model = d_model
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

    self.layers = nn.ModuleList(
        [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
    )

    # Pre norm requires this as the residual stream is never normalized inside the layers, so without a final norm here, values could grow unbounded in scale
    self.final_norm = nn.LayerNorm(d_model)

  def forward(self, src, src_mask):
    # Here we are rescalling the magnitude of embeddings as they must be of comparable magnitudes
    x = self.embedding(src) * math.sqrt(self.d_model)
    x = self.pos_encoding(x)

    for layer in self.layers:
      x = layer(x, src_mask)

    return self.final_norm(x)  # (B, src_len, d_model) -- fed into every decoder layer's cross-attention


## Decoder (Stack of N Decoder Layers/Blocks)

In [39]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
    super().__init__()

    self.d_model = d_model
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

    self.layers = nn.ModuleList(
        [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
    )

    self.final_norm = nn.LayerNorm(d_model) # Same reason as given in the encoder block
    self.output_proj = nn.Linear(d_model, vocab_size) # final linear -> logits for Hindi vocab

  def forward(self, tgt, enc_output, src_mask, tgt_mask):
    x = self.embedding(tgt) * math.sqrt(self.d_model)
    x = self.pos_encoding(x)

    for layer in self.layers:
      x = layer(x, enc_output, src_mask, tgt_mask)

    return self.output_proj(self.final_norm(x)) # (B, tgt_len, vocab_size) -> raw logits, softmax will be applied in the loss function

## Full transformer

In [40]:
class Transformer(nn.Module):
  def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, num_layers=3, num_heads=4, d_ff=1024, max_len=64, dropout=0.1, pad_id=0):
    super().__init__()

    self.pad_id = pad_id
    self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
    self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)

    # Xavier weight initialization
    for p in self.parameters():
      if p.dim() > 1:
        nn.init.xavier_uniform_(p)

  def make_src_mask(self, src):
    # src - (B, src_len)
    # output - (B, 1, 1, src_len) so it broadcasts over heads and query positions
    mask = (src != self.pad_id).unsqueeze(1).unsqueeze(2)
    return mask

  def make_tgt_mask(self, tgt):
    # Combines padding mask AND causal mask into one.
    # tgt: (B, tgt_len) token ids
    B, tgt_len = tgt.shape

    pad_mask = (tgt != self.pad_id).unsqueeze(1).unsqueeze(2) # (B, 1, 1, tgt_len)

    causal_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool() # (tgt_len, tgt_len)
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0) # (1, 1, tgt_len, tgt_len)

    # Both conditions must hold: real tokens AND not looking into the future
    tgt_mask = pad_mask & causal_mask

    return tgt_mask

  def forward(self, src, tgt):
    src_mask = self.make_src_mask(src) # (B, 1, 1, src_len)
    tgt_mask = self.make_tgt_mask(tgt) # (B, 1, tgt_len, tgt_len)

    enc_output = self.encoder(src, src_mask)
    logits = self.decoder(tgt, enc_output, src_mask, tgt_mask)

    return logits


In [41]:
# Sanity check
SRC_VOCAB = 8000
TGT_VOCAB = 8000
BATCH = 4
SRC_LEN = 20
TGT_LEN = 15

model = Transformer(SRC_VOCAB, TGT_VOCAB, d_model=256, num_layers=3, num_heads=4, d_ff=1024, max_len=64)

dummy_src = torch.randint(1, SRC_VOCAB, (BATCH, SRC_LEN))
dummy_tgt = torch.randint(1, TGT_VOCAB, (BATCH, TGT_LEN))

logits = model(dummy_src, dummy_tgt)
print("output logits shape:", logits.shape)  # expect (4, 15, 8000)

num_params = sum(p.numel() for p in model.parameters())
print(f"total params: {num_params:,}")

output logits shape: torch.Size([4, 15, 8000])
total params: 11,682,624


# Training loop

## Setup

In [42]:
import math
import time
import torch
import torch.nn as nn

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [44]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [45]:
CHECKPOINT_DIR = "/content/drive/MyDrive/en_hi_transformer/checkpoints"

In [46]:
import os; os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [47]:
model = Transformer(
    src_vocab_size=VOCAB_SIZE_EN,
    tgt_vocab_size=VOCAB_SIZE_HI,
    d_model=256,
    num_layers=3,
    num_heads=4,
    d_ff=1024,
    max_len=MAX_SEQ_LEN,
    dropout=0.1,
    pad_id=PAD_ID,
).to(device)

## Loss function: (Masked cross-entropy)

* Here we will use ignore index = pad_id, this is important because the model will be punished with loss if it does not predict the pad properly otherwise which is some unwanted data that we do not want from the final model and the pad's are batch dependent and random to the model. It is meaningless loss that will hurt the training.
* Here we are using label_smoothing set to 0.1 as this would not let the model choose one particular token with 100 percent confidence and spread around the percentage more. This would help the model from refraining overconfidence and find more choices.

In [48]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

## Optimizer + Learning Rate Scheduler

Original paper uses a linear ramp in LR and then a proportional decay of 1/sqrt(step) after a particular amount of epochs (here 4000).

Warmup matters alot in transformer training, this is because Adam initially does not have any idea of the averages and can overshoot the gradients and destabilise the model to the point where it will not train good, so it is better to first use really small LR and let the optimizer understand the general direction in which the model must head for better loss. Then a linear ramp up to that direction would help reach the minima following with a decay that slowly helps the model stabilise in that minima rather than bouncing around.

In [49]:
class WarmupScheduler:
  def __init__(self, optimizer, d_model, warmup_steps=4000):
    self.optimizer = optimizer
    self.d_model = d_model
    self.warmup_steps = warmup_steps
    self.step_num = 0

  def step(self):
    self.step_num += 1
    lr = self._compute_lr()
    for param_group in self.optimizer.param_groups:
      param_group['lr'] = lr
    self.optimizer.step()

  def _compute_lr(self):
    step = max(self.step_num, 1)
    return (self.d_model ** -0.5) * min(step ** -0.5, step * self.warmup_steps ** -1.5)

In [50]:
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = WarmupScheduler(optimizer, d_model=256)

## Training step

In [51]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, max_grad_norm=1.0, log_every=50):
  model.train()
  total_loss = 0.0
  total_tokens = 0

  for step, token in enumerate(loader):
    src = batch['src'].to(device)
    tgt_in = batch['tgt_in'].to(device)
    tgt_out = batch['tgt_out'].to(device)

    optimizer.zero_grad()

    logits = model(src, tgt_in) # (B, tgt_len, vocab_size)

    # CrossEntropyLoss expects (N, C) vs (N,) -- flatten batch and seq_len together
    loss = criterion(
        logits.view(-1, logits.shape[-1]), # (B * tgt_len, vocab_size)
        tgt_out.view(-1)                   # (B * tgt_len, )
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    optimizer.step()
    scheduler.step()

    # Tracking loss weighted by the number of real tokens
    # not by batch count as batches have different number of real tokens due to bucketing
    # so simply average-of-batch-losses would be slightly biased.
    # This gives a more honest per token loss
    num_real_tokens = (tgt_out != PAD_ID).sum().item()
    total_loss += loss.item() * num_real_tokens
    total_tokens += num_real_tokens

    if step % log_every == 0:
      current_lr = optimizer.param_groups[0]['lr']
      print(f"  step {step:5d} | loss {loss.item():.4f} | lr {current_lr:.6f}")

  return total_loss / total_tokens

## Validation step

In [52]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss = 0.0
  total_tokens = 0

  for step, batch in enumerate(loader):
    src = batch['src'].to(device)
    tgt_in = batch['tgt_in'].to(device)
    tgt_out = batch['tgt_out'].to(device)

    logits = model(src, tgt_in)

    loss = criterion(
      logits.view(-1, logits.shape[-1]),
      tgt_out.view(-1)
    )

    num_real_tokens = (tgt_out != PAD_ID).sum().item()
    total_loss += loss.item() * num_real_tokens
    total_tokens += num_real_tokens

  return total_loss / total_tokens

## Checkpointing

This is an important step as colab notebook will disconnect and the T4 runtime will keep exhausting. So will be saving the model's updates every once in a while during training to resume once the runtime is available again. ;-;

In [53]:
def save_checkpoint(model, optimizer, scheduler, epoch, path):
  torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_step_num": scheduler.step_num,
  }, path)
  print(f"checkpoint saved: {path}")

In [54]:
def load_checkpoint(model, optimizer, scheduler, path, device):
  ckpt = torch.load(path, map_location=device)
  model.load_state_dict(ckpt["model_state_dict"])
  optimizer.load_state_dict(ckpt["optimizer_state_dict"])
  scheduler.step_num = ckpt["scheduler_step_num"]
  start_epoch = ckpt["epoch"] + 1
  print(f"resumed from {path}, starting at epoch {start_epoch}")
  return start_epoch

## Debug run

This is important to see if the loss is actually reducing for a smaller dataset for a small number of epochs before actually using the bigger dataset with multiple GPU hours

In [55]:
from torch.utils.data import Subset

In [56]:
debug_dataset = Subset(train_dataset, range(500))
debug_loader = DataLoader(debug_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [57]:
print('running debug overfit test on 500 examples')
for epoch in range(10):
  train_loss = train_one_epoch(model, debug_loader, optimizer, scheduler, criterion, device, log_every=10)
  print(f"debug epoch {epoch}: loss = {train_loss:.4f}")

running debug overfit test on 500 examples
  step     0 | loss 9.0119 | lr 0.000000
  step    10 | loss 9.0083 | lr 0.000003
  step    20 | loss 8.9380 | lr 0.000005
  step    30 | loss 8.8361 | lr 0.000008
debug epoch 0: loss = 8.9417
  step     0 | loss 8.8122 | lr 0.000008
  step    10 | loss 8.7003 | lr 0.000011
  step    20 | loss 8.5578 | lr 0.000013
  step    30 | loss 8.4359 | lr 0.000016
debug epoch 1: loss = 8.6206
  step     0 | loss 8.3865 | lr 0.000016
  step    10 | loss 8.2408 | lr 0.000019
  step    20 | loss 8.0662 | lr 0.000021
  step    30 | loss 7.8507 | lr 0.000023
debug epoch 2: loss = 8.1313
  step     0 | loss 7.8131 | lr 0.000024
  step    10 | loss 7.5669 | lr 0.000026
  step    20 | loss 7.2908 | lr 0.000029
  step    30 | loss 7.0224 | lr 0.000031
debug epoch 3: loss = 7.4172
  step     0 | loss 6.9593 | lr 0.000032
  step    10 | loss 6.6766 | lr 0.000034
  step    20 | loss 6.3809 | lr 0.000037
  step    30 | loss 6.0701 | lr 0.000039
debug epoch 4: loss =

## Inference (Greedy Decoding)

Assumes model.py has run and you have a trained `model`

This helps ensure that all the checkpointing and other tools are working as desired before the final proper run.

In [58]:
import torch

In [60]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

### Greedy decode

(token ids in -> token ids out)

In [69]:
@torch.no_grad()
def greedy_decode(model, src, max_len, device, sos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID):
  # src -> (1, src_len)
  # Using batch size as 1 which is not needed for a first working version
  model.eval()
  src = src.to(device)

  # Computing the encoder output once as the english sentence is not changing
  # Only the hindi output tokens will be changing for the same input english tokens
  # It is better than wasting time by redoing the same work unnecesarily
  src_mask = model.make_src_mask(src)
  enc_output = model.encoder(src, src_mask) # (1, src_len, d_model)

  # Starting the decoder output using only the <bos> token
  tgt_ids = [sos_id]

  for _ in range(max_len):
    tgt_tensor = torch.tensor([tgt_ids], device=device) # (1, current_len)
    tgt_mask = model.make_tgt_mask(tgt_tensor) # (1, 1, current_len, current_len) This is done every step as the length of tensor grows every iteration

    logits = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask) # (1, current_len, vocab_size)

    # We only care about the prediction of the next token
    # which is at the last position of the current sequence
    next_token_logits = logits[0, -1, :] # (vocab_size, )
    next_token_id = next_token_logits.argmax(dim=-1).item() # greedy: always pick the single most likely token

    tgt_ids.append(next_token_id)

    if next_token_id == eos_id:
      break

  # Striping the <bos> token and possible <eos> token
  output_ids = tgt_ids[1:]
  if output_ids and output_ids[-1] == eos_id:
    output_ids = output_ids[:-1]

  return output_ids

### End-to-end raw english -> raw hindi convertion

In [70]:
def translate(model, sentence, sp_en, sp_hi, max_len, device):
  src_ids = sp_en.encode(sentence, out_type=int)
  src_ids = src_ids[: max_len - 1] + [EOS_ID] # same convention as training data

  src_tensor = torch.tensor([src_ids], device=device) # (1, src_len) -> batch dim 1

  output_ids = greedy_decode(model, src_tensor, max_len, device)

  hindi_text = sp_hi.decode(output_ids)
  return hindi_text

In [74]:
translate(model, "what is your name?", sp_en, sp_hi, 50, device)

'I रूपरेखा रूपरेखा रूपरेखा रूपरेखा रूपरेखा जानकारी जानकारी जानकारी I I I I I I I I I I I I I I I I I I जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी ध्यान रूपरेखा जानकारी जानकारी जानकारी I I I I I I I'

### Batch translate

In [76]:
def translate_many(model, sentences, sp_en, sp_hi, max_len, device):
  return [translate(model, s, sp_en, sp_hi, max_len, device) for s in sentences]

In [78]:
translate_many(model, ['what is your age', 'where do you work', 'i love sports', 'this is fun'], sp_en, sp_hi, 20, device)

['I रूपरेखा रूपरेखा रूपरेखा रूपरेखा रूपरेखा जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी',
 '्रा ता नया नयाMF नयाMFMFMFMFMF ताMFusus दस्ती ताususus',
 'ध्यान ध्यान प्रकार प्रकार प्रकार जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी',
 'ध्यान ध्यान रूपरेखा रूपरेखा रूपरेखा रूपरेखा रूपरेखा रूपरेखा जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी']

In [96]:
sentences_from_debug_dataset = [sp_en.decode(tokens.tolist()) for tokens in next(iter(debug_loader))['src']]
for sentence in sentences_from_debug_dataset:
  print(sentence)

Block Ten
Include defaults
Window height value.
_ Contents
INFO
A list of strings that come in the form of a quintuple: name, wins, total games played, best time (in seconds) and worst time (also in seconds). Unplayed games do not need to be represented.
Select the game type to play
Script Type
API Browser
Helsinki
Everything
You should have received a copy of the GNU General Public License along with this program. If not, see.
0, 0
Bottom panel
Help
Selected accessible


In [95]:
translate_many(model, sentences_from_debug_dataset, sp_en, sp_hi, 20, device)

['1 Iमेंमेंमेंमेंमें I जानकारी जानकारी I I I I I जानकारी जानकारी जानकारी जानकारी जानकारी',
 'IIT प्रस्थिति I I जानकारी जानकारी जानकारी I I I I I I I I I I I I',
 'ध्याननेटिटिटिटिटिटिटिटिटिटिटिटिटिटिटिटिटिटि',
 'I ध्यान दिखानामेंमेंमेंमें I I I I I I I I I I I I I',
 'ओंओंओंओंओंओंओंओंओंओंओंओंओंओंओंओंओंओंओं कृपया',
 'खो I पह मौजूदा वा वा मौजूदा पह मौजूदा पह देखें वा I वा वा वा वा I देखें देखें',
 'I I I I I I I I I I I I I I I I I I I I',
 'I I दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना I I I I I I I I I जानकारी जानकारी जानकारी',
 'I I I I I I I I I I I I I I I I I I I देखें',
 'नाम्या्या्या्या्या्या्या्या्या्या्या्या्या्या्या्या्या्या्या',
 'सॉलिटेयरमेंमेंमेंमेंमेंमेंमें दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने दिखाने',
 'देखें पह पह पह पह पह पह पह पह पह पह पह पह पह पह पह देखें पह देखें देखें',
 'I दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना दिखाना जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी जानकारी',
 'देखें देखें 

### Contamination Test

As we can see that the hindi output has a few other tokens like 'I' or 'us' which are probably there due to the contaminated dataset issues. So to ensure that the percentage of contamination is less, the following code is given

If the contamination is alot, will have to purify the dataset on loading

In [97]:
import re

In [100]:
def latin_char_ratio(line):
  # Fraction of alphabetic characters in the line that are Latin rather than Devanagari or anything else
  alpha_chars = [c for c in line if c.isalpha()]
  if not alpha_chars:
    return 0.0
  latin_chars = [c for c in alpha_chars if ord(c) < 0x0900]  # below Devanagari unicode block
  return len(latin_chars) / len(alpha_chars)

In [101]:
def check_file(path, threshold=0.3, sample_print=5):
  with open(path, encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

  contaminated = []
  for line in lines:
    ratio = latin_char_ratio(line)
    if ratio > threshold:
      contaminated.append((ratio, line))

  print(f"total lines: {len(lines)}")
  print(f"lines with >{threshold*100:.0f}% Latin characters: {len(contaminated)} ({100*len(contaminated)/len(lines):.2f}%)")
  print()
  print(f"sample contaminated lines:")
  for ratio, line in sorted(contaminated, key=lambda x: -x[0])[:sample_print]:
    print(f"  [{ratio:.2f}] {line[:100]}")

In [102]:
check_file('data/train.hi')

total lines: 21588
lines with >30% Latin characters: 2078 (9.63%)

sample contaminated lines:
  [1.00] % s is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the im
  [1.00] Acard symbol
  [1.00] 2number
  [1.00] 3number
  [1.00] 4number


### Fixes:

Will have to update the data loading to remove the contamination.

Also here during the debug test, the model never left the warmup ever. So the model did not learn the data as well as it could have when the learning rate would have peaked.

Plus, the raw corpus has alot of data from software UI strings, subtitles, news, etc. So the data requires to be pre shuffled before being split into the train, val and test datasets